# Fig. 2g — Metabolic profiles of VHC subtypes (scCellFie)

Per-subtype radial plot of scaled metabolic-task activity inferred by [scCellFie](https://github.com/earmingol/scCellFie).

**Two stages**
1. **Run pipeline** — `sccellfie.run_sccellfie_pipeline` on the VHC subset; writes `sccellfie_results/Vestibular_VHC_scCellFie_metabolic_tasks.h5ad`. This is the expensive step (smoothing, threshold inference, GPR resolution).
2. **Radial plot** — load saved metabolic-task adata, aggregate per VHC subtype (`trimean`), min-max normalise per task across subtypes, plot one radial panel per subtype.

**Inputs**
- `Vestibular_VGN_VHC_merged.h5ad` (pipeline stage)
- `sccellfie_results/Vestibular_VHC_scCellFie_metabolic_tasks.h5ad` (plot stage, generated by stage 1)

**Output**
- `figures/Fig2g_VHC_sccellfie_radial.pdf`

In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import sccellfie
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1
sc.set_figure_params(figsize=(5, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

SCCELLFIE_DIR = DATA_DIR / 'sccellfie_results'
SCCELLFIE_DIR.mkdir(exist_ok=True)
METABOLIC_TASKS_H5AD = SCCELLFIE_DIR / 'Vestibular_VHC_scCellFie_metabolic_tasks.h5ad'

## Stage 1 — Run pipeline (skip if `Vestibular_VHC_scCellFie_metabolic_tasks.h5ad` already exists)

In [ ]:
if not METABOLIC_TASKS_H5AD.exists():
    adata_all = sc.read_h5ad(DATA_DIR / 'Vestibular_VGN_VHC_merged.h5ad')
    adata_HC_ss3 = adata_all[adata_all.obs['batch'].isin(['HC_ss3'])].copy()
    adata_HC_ss3.X = adata_HC_ss3.layers['umi'].copy()

    # standard CPM-like normalisation; scCellFie uses log1p-scaled values for GPR resolution
    sc.pp.normalize_total(adata_HC_ss3)
    sc.pp.log1p(adata_HC_ss3)

    results = sccellfie.run_sccellfie_pipeline(
        adata_HC_ss3,
        organism='mouse',
        sccellfie_data_folder=None,
        n_counts_col=None,
        process_by_group=False,
        groupby=None,
        neighbors_key='neighbors',
        n_neighbors=20,
        batch_key=None,
        threshold_key='sccellfie_threshold',
        smooth_cells=True,
        alpha=0.33,
        chunk_size=5000,
        disable_pbar=False,
        save_folder=str(SCCELLFIE_DIR),
        save_filename='Vestibular_VHC_scCellFie',
    )
    print('Pipeline finished. Saved to', METABOLIC_TASKS_H5AD)
else:
    print('Found existing metabolic_tasks h5ad, skipping pipeline:')
    print(' ', METABOLIC_TASKS_H5AD)

## Stage 2 — Radial plot of scaled metabolic-task activity per VHC subtype

In [ ]:
metabolic_tasks_adata = sc.read_h5ad(METABOLIC_TASKS_H5AD)

# task_info table (Task -> System / Subsystem) is deterministic for a given organism
db = sccellfie.datasets.load_sccellfie_database(organism='mouse')
task_info = db['task_info']
print('metabolic_tasks:', metabolic_tasks_adata.shape)
print('task_info     :', task_info.shape)

In [ ]:
# Aggregate cell-level task activity to per-subtype activity via trimean (robust central tendency),
# then min-max normalise per task across subtypes so each task contributes equally to the radial.
agg = sccellfie.expression.aggregation.agg_expression_cells(
    metabolic_tasks_adata,
    groupby='all_cluster_gene_name',
    agg_func='trimean',
)
input_df = sccellfie.preprocessing.matrix_utils.min_max_normalization(agg.T, axis=1)
print('Tasks x Subtypes matrix:', input_df.shape)

In [ ]:
# Reshape to long-format with System annotation (required by sccellfie.plotting.create_radial_plot)
df_melted = pd.melt(
    input_df.reset_index(),
    id_vars='Task',
    var_name='cell_type',
    value_name='scaled_trimean',
).rename(columns={'Task': 'metabolic_task'})

df_melted = df_melted.merge(
    task_info.set_index('Task')[['System']],
    left_on='metabolic_task', right_index=True, how='left',
)

In [ ]:
cell_types = df_melted['cell_type'].unique().tolist()
ncols = 3
nrows = math.ceil(len(cell_types) / ncols)

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    subplot_kw={'projection': 'polar'},
    figsize=(5 * ncols, 5 * nrows),
)
axes = axes.flatten()

for i, (cell, ax) in enumerate(zip(cell_types, axes)):
    sccellfie.plotting.create_radial_plot(
        df_melted, task_info,
        cell_type=cell, ax=ax,
        show_legend=(i == 0),
        ylim=1.0, sort_by_value=True,
    )

for j in range(len(cell_types), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.savefig(FIG_DIR / 'Fig2g_VHC_sccellfie_radial.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()